In [1]:
import copy
import json
import numbers
import os
import sys
import time
from datetime import datetime, timezone
from functools import partial
from pathlib import Path
from types import SimpleNamespace

# Read when torch first touches the GPU, so it only takes effect in a fresh kernel. Without it the GiB-sized blocks of the
# contrastive loss fragment GPU memory and batch 1024 runs out of memory on the click graph (peak 10.5 GiB with it).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import kagglehub
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
import scipy.sparse as sp
import torch
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "lightgcl.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from lightgcl import LightGCLTimeSoftmax  # noqa: E402
from utils import normalize_adj, svd_view, to_torch_sparse, topk_metrics, train_epoch  # noqa: E402

DATA_DIR = Path(
    os.environ.get("EXPEDIA_DATA_DIR")
    or kagglehub.competition_download("expedia-hotel-recommendations")
)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def fmt_num(x, digits=4):
    """37670293 -> '37.670.293', 0.12345 -> '0,1235' (at most `digits` decimals)."""
    text = f"{x:,.0f}" if float(x).is_integer() else f"{x:,.{digits}f}".rstrip("0").rstrip(".")
    return text.translate(str.maketrans(",.", ".,"))


def fmt_pct(x, digits=2):
    """0.359036 -> '35,90%'."""
    return f"{x:.{digits}%}".replace(".", ",")


def fmt_table(df):
    """Display copy of df: numbers in Vietnamese format, columns whose name contains '%' as percentages."""
    def cell(value, pct):
        if isinstance(value, bool) or not isinstance(value, numbers.Number):
            return value
        if pd.isna(value):
            return "—"
        return fmt_pct(value) if pct else fmt_num(value)

    out = df.astype(object)
    for col in df.columns:
        out[col] = [cell(value, "%" in str(col)) for value in df[col]]
    return out

In [10]:
cfg = SimpleNamespace(
    item_cols=["hotel_cluster"],
    val_start="2014-09-01",
    test_start="2014-11-01",
    test_end="2015-01-01",
    embed_dim=64,
    n_layers=2,
    temp=0.2,
    svd_q=2,
    svd_exact=True,
    lambda_1=0.2,
    lambda_2=1e-4,
    dropout=0.0,
    lr=1e-4,  # starting lr; ReduceLROnPlateau multiplies it by lr_factor after each epoch without a better val main_metric
    lr_factor=0.3,
    lr_patience=0,
    lr_min=1e-6,
    batch_size=1024,
    epochs=20,
    extend_epochs=10,  # a run still improving at its epochs cap (early stopping not fired) goes on this many epochs at a time
    patience=3,
    main_metric="recall@5",
    topks=(5, 10, 20),
    seed=2026,
    # Run directory in data/runs: the model saved there is reused; when there is none, the model is trained and saved there.
    run="lightgcl_final_timeattn",
    # The two earlier LightGCL models put on the test table next to it, trained in LightGCL_Click.ipynb whose model.pt
    # holds one state per variant: label -> (run, variant, edge weights: "binary" 1 per booked pair, "count" its number
    # of bookings)
    old_models={
        "LightGCL, cạnh 0–1": ("lightgcl_click_20261004_115153", "LightGCL, chỉ booking", "binary"),
        "LightGCL, cạnh theo số lần book": ("lightgcl_click_20261004_115153",
                                            "LightGCL, booking theo số lần, λ2 = 1e-5", "count"),
    },
)


In [3]:
COLUMN_TYPES = {"date_time": pa.timestamp("s"), "user_id": pa.int32(), "is_booking": pa.int8(),
                **{c: pa.int32() for c in cfg.item_cols}}
events = pv.read_csv(
    DATA_DIR / "train.csv",
    convert_options=pv.ConvertOptions(include_columns=list(COLUMN_TYPES), column_types=COLUMN_TYPES),
).to_pandas()
val_start, test_start, test_end = (pd.Timestamp(d) for d in (cfg.val_start, cfg.test_start, cfg.test_end))
events = events[events["date_time"] < test_end]
events["item"] = events.groupby(cfg.item_cols, sort=True).ngroup().astype(np.int32)
n_items = events["item"].nunique()
bookings = events[events["is_booking"] == 1]


def count_matrix(rows, user_index):
    """users x items number of events in rows."""
    pair_counts = rows.groupby([user_index.get_indexer(rows["user_id"]), rows["item"]]).size()
    return sp.csr_matrix(
        (pair_counts.to_numpy(np.float32),
         (pair_counts.index.get_level_values(0), pair_counts.index.get_level_values(1))),
        shape=(len(user_index), n_items),
    )


def booking_graph(end):
    """Graph of the bookings before `end`: train_csr holds the booked pairs, the positives of the loss; adj_csr the same
    pairs weighted by their number of bookings, the edges the GNN and the SVD view propagate over; the time attention
    weights are built from the bookings before `end` as well."""
    rows = bookings[bookings["date_time"] < end]
    user_index = pd.Index(np.sort(rows["user_id"].unique()), name="user_id")
    book_counts = count_matrix(rows, user_index)
    return SimpleNamespace(end=end, user_index=user_index, n_users=len(user_index), book_counts=book_counts,
                           train_csr=(book_counts > 0).astype(np.float32).tocsr(), adj_csr=book_counts)


def make_split(start, end):
    """Labels of the bookings in [start, end): user ids and one row per user, one column per cluster."""
    pairs = bookings[bookings["date_time"].between(start, end, inclusive="left")][["user_id", "item"]].drop_duplicates()
    users, row = np.unique(pairs["user_id"].to_numpy(), return_inverse=True)
    return SimpleNamespace(
        user_ids=users,
        labels=sp.csr_matrix((np.ones(len(pairs), np.float32), (row, pairs["item"].to_numpy())),
                             shape=(len(users), n_items)),
    )


GRAPHS = {"train": booking_graph(val_start), "train+val": booking_graph(test_start)}  # before 09/2014, before 11/2014
val_split = make_split(val_start, test_start)
test_split = make_split(test_start, test_end)

test_users = pd.Index(test_split.user_ids)
old_users = GRAPHS["train+val"].user_index
USER_GROUPS = {
    "Tất cả user test": test_users,
    "User cũ (đã book trước 11/2014)": old_users,
    "User đã book trước 09/2014": GRAPHS["train"].user_index,
    "User book lần đầu ở 09–10/2014": old_users.difference(GRAPHS["train"].user_index),
    "User mới (chưa book trước 11/2014)": test_users.difference(old_users),
}
display(fmt_table(pd.DataFrame({
    label: {"user": g.n_users, "cặp (user, cluster) đã book": g.train_csr.nnz, "lượt booking": g.adj_csr.sum()}
    for label, g in GRAPHS.items()
}).T))
group_sizes = pd.Series({group: np.isin(test_split.user_ids, users).sum() for group, users in USER_GROUPS.items()})
display(fmt_table(pd.DataFrame({"user test": group_sizes, "% user test": group_sizes / len(test_users)})))

,user,"cặp (user, cluster) đã book",lượt booking
train,664.727,1.776.919,2.214.809
train+val,742.268,2.075.812,2.615.006


,user test,% user test
Tất cả user test,222.688,"100,00%"
User cũ (đã book trước 11/2014),150.971,"67,79%"
User đã book trước 09/2014,135.589,"60,89%"
User book lần đầu ở 09–10/2014,15.382,"6,91%"
User mới (chưa book trước 11/2014),71.717,"32,21%"


## 1. Đánh giá

In [4]:
METRICS = [f"{m}@{k}" for m in ("recall", "ndcg", "map") for k in cfg.topks]
REPORT = ["recall@5", "recall@20", "ndcg@5", "ndcg@20", "map@5", "map@20"]  # the columns of the result tables
EVAL_BATCH = max(256, 2**24 // n_items)


def to_dense(csr_rows):
    return torch.as_tensor(csr_rows.toarray(), device=DEVICE)


def popular(graph):
    """Clusters ranked by the number of users of graph who booked them, the same list for every user."""
    pop = torch.as_tensor(np.asarray(graph.train_csr.sum(0)).ravel(), device=DEVICE)
    return lambda user_ids: pop.expand(len(user_ids), -1)


def with_fallback(score_fn, graph):
    """score_fn for the users of graph; the users it does not hold (cold start) get the Popular list of graph."""
    fallback = popular(graph)

    def scores(user_ids):
        known = np.isin(user_ids, graph.user_index)
        out = fallback(user_ids).float().clone()
        if known.any():
            out[torch.as_tensor(known, device=DEVICE)] = score_fn(user_ids[known]).float()
        return out
    return scores


def history(graph):
    """Clusters the user booked first, by number of bookings, then the others; popularity (scaled below 1) breaks ties."""
    pop = torch.as_tensor(np.asarray(graph.train_csr.sum(0)).ravel(), device=DEVICE)
    return with_fallback(
        lambda user_ids: to_dense(graph.book_counts[graph.user_index.get_indexer(user_ids)]) + pop / (pop.max() + 1),
        graph)


@torch.no_grad()
def evaluate(score_fn, split, users):
    """Mean metrics over the users of `split` that are in `users` (an Index of user ids); score_fn maps an array of
    user ids to a (users x items) score tensor."""
    keep = np.isin(split.user_ids, users)
    user_ids, labels = split.user_ids[keep], split.labels[keep]
    parts = []
    for start in range(0, len(user_ids), EVAL_BATCH):
        scores = score_fn(user_ids[start:start + EVAL_BATCH]).float()
        batch_labels = to_dense(labels[start:start + EVAL_BATCH]) > 0
        parts.append({k: v.cpu() for k, v in topk_metrics(scores, batch_labels, cfg.topks).items()})
    return pd.Series({m: torch.cat([p[m] for p in parts]).mean().item() for m in METRICS})

## 2. Training

In [12]:
def time_attention(graph):
    """Time attention weights of a graph, built as in LightGCL_Click.ipynb: the softmax over each user's clusters (time_u,
    users x items) and over each cluster's users (time_i, items x users) of the pair's last booking time, scaled to
    [0, 1] over the graph."""
    rows = bookings[bookings["date_time"] < graph.end]
    last = rows.groupby([graph.user_index.get_indexer(rows["user_id"]), rows["item"].to_numpy()])["date_time"].max()
    scaled = ((last - last.min()) / (last.max() - last.min())).to_numpy(np.float64)
    exp_t = sp.csr_matrix((np.exp(scaled), (last.index.get_level_values(0), last.index.get_level_values(1))),
                          shape=(graph.n_users, n_items))
    time_u = sp.diags(1 / np.asarray(exp_t.sum(1)).ravel()) @ exp_t
    time_i = (exp_t @ sp.diags(1 / np.asarray(exp_t.sum(0)).ravel())).T
    return time_u, time_i


def adjacency(graph, edges):
    """User side (users x items) and item side (items x users) of the adjacency a model propagates over on graph: the
    normalized adjacency of the booked pairs, each weighing 1 (edges "binary") or its number of bookings ("count");
    "time" (LightGCLTimeSoftmax) averages the latter with the time attention weights."""
    adj = normalize_adj(graph.train_csr if edges == "binary" else graph.adj_csr).tocsr()
    if edges != "time":
        return adj, adj.T
    time_u, time_i = time_attention(graph)
    return 0.5 * adj + 0.5 * time_u, 0.5 * adj.T + 0.5 * time_i


def make_model(graph):
    """LightGCL + time attention on graph, built as in LightGCL_Click.ipynb: the SVD view and the contrastive loss use
    the normalized adjacency, the graph view also the time attention weights."""
    adj_norm = to_torch_sparse(normalize_adj(graph.adj_csr), DEVICE)
    torch.manual_seed(cfg.seed)  # svd_lowrank is randomized
    factors = svd_view(adj_norm, cfg.svd_q, cfg.svd_exact)
    time_u, time_i = (to_torch_sparse(m.tocoo(), DEVICE) for m in time_attention(graph))
    torch.manual_seed(cfg.seed)  # the same initial embeddings as in LightGCL_Click.ipynb
    return LightGCLTimeSoftmax(graph.n_users, n_items, cfg.embed_dim, *factors, graph.train_csr, adj_norm, cfg.n_layers,
                               cfg.temp, cfg.lambda_1, cfg.lambda_2, cfg.dropout, time_u=time_u, time_i=time_i).to(DEVICE)


def lightgcl_scorer(model, graph):
    # Raw scores: clusters the user already booked stay in the ranking, as for the baselines.
    return lambda user_ids: model.predict(torch.as_tensor(graph.user_index.get_indexer(user_ids), device=DEVICE),
                                          exclude_seen=False)


In [6]:
def train_model(graph):
    """Train LightGCL + time attention on graph: early stopping on the validation main_metric with ReduceLROnPlateau; the
    best epoch is kept. Still improving at the epochs cap: goes on extend_epochs at a time."""
    label = "LightGCL + time attention"
    model = make_model(graph)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=0)
    # threshold 0: the lr drops exactly when the epoch is not marked as an improvement below
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=cfg.lr_factor,
                                                           patience=cfg.lr_patience, threshold=0, min_lr=cfg.lr_min)
    gen = torch.Generator(device=DEVICE).manual_seed(cfg.seed)
    coo = graph.train_csr.tocoo()
    edge_users = torch.as_tensor(coo.row, dtype=torch.long, device=DEVICE)
    edge_items = torch.as_tensor(coo.col, dtype=torch.long, device=DEVICE)
    score_fn = lightgcl_scorer(model, graph)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
    history, best = [], {"epoch": 0, "score": -np.inf, "state": None}
    epoch, cap = 0, cfg.epochs
    while epoch < cap and epoch - best["epoch"] < cfg.patience:
        epoch += 1
        t0, lr = time.time(), optimizer.param_groups[0]["lr"]
        loss, loss_r, loss_s = train_epoch(
            model, optimizer, edge_users, edge_items, n_items, cfg.batch_size, gen,
            progress=partial(tqdm, desc=f"{label} · epoch {epoch}", leave=False),
        )
        if not np.isfinite(loss):
            raise FloatingPointError(f"[{label}] loss = {loss} ở epoch {epoch}")
        train_seconds = time.time() - t0
        val = evaluate(score_fn, val_split, graph.user_index)
        improved = val[cfg.main_metric] > best["score"]
        scheduler.step(val[cfg.main_metric])
        if improved:
            best = {"epoch": epoch, "score": val[cfg.main_metric],
                    "state": {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}}
        history.append({"epoch": epoch, "lr": lr, "loss": loss, "loss_r": loss_r, "loss_s": loss_s,
                        "giây train": train_seconds, **val.to_dict()})
        print(f"[{label}] epoch {epoch:>2} | lr {lr:.0e} | loss {loss:.4f} (rec {loss_r:.4f}, cl {loss_s:.4f}) | "
              f"val {cfg.main_metric} {val[cfg.main_metric]:.4f}{' *' if improved else ''} | {train_seconds:.0f}s train")
        if epoch == cap and epoch - best["epoch"] < cfg.patience:
            print(f"[{label}] chưa hội tụ ở epoch {epoch}: train tiếp {cfg.extend_epochs} epoch")
            cap += cfg.extend_epochs
    return SimpleNamespace(best=best, history=pd.DataFrame(history).set_index("epoch"),
                           peak_gib=torch.cuda.max_memory_allocated(DEVICE) / 2**30 if DEVICE.type == "cuda" else np.nan)


run_dir = PROJECT_ROOT / "data" / "runs" / cfg.run
if (run_dir / "model.pt").is_file():
    saved = torch.load(run_dir / "model.pt", map_location="cpu", weights_only=False)
    train_history = pd.read_csv(run_dir / "history.csv", index_col="epoch")
    print(f"Dùng lại model đã train trong {run_dir.relative_to(PROJECT_ROOT)}")
else:
    r = train_model(GRAPHS["train"])
    saved = {"state": r.best["state"], "user_ids": GRAPHS["train"].user_index.to_numpy(),
             "result": {"epoch tốt nhất": r.best["epoch"], f"val {cfg.main_metric}": float(r.best["score"]),
                        "epoch đã train": len(r.history), "giây train / epoch": float(r.history["giây train"].mean()),
                        "GPU đỉnh (GiB)": r.peak_gib}}
    train_history = r.history
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(saved, run_dir / "model.pt")
    train_history.to_csv(run_dir / "history.csv")
    (run_dir / "config.json").write_text(json.dumps({**vars(cfg), "result": saved["result"]}, ensure_ascii=False,
                                                    indent=2, default=float))
    print(f"Đã lưu tham số epoch {r.best['epoch']} vào {run_dir.relative_to(PROJECT_ROOT)}/model.pt")
    del r
    torch.cuda.empty_cache()
display(fmt_table(train_history[["lr", "loss", "loss_r", "loss_s", *REPORT]]))
display(fmt_table(pd.DataFrame({"LightGCL + time attention": saved["result"]}).T))


LightGCL + time attention · epoch 1:   0%|          | 0/1736 [00:00<?, ?it/s]

[LightGCL + time attention] epoch  1 | lr 1e-04 | loss 10.7164 (rec 2.9036, cl 7.5946) | val recall@5 0.2185 * | 222s train


LightGCL + time attention · epoch 2:   0%|          | 0/1736 [00:00<?, ?it/s]

[LightGCL + time attention] epoch  2 | lr 1e-04 | loss 6.5253 (rec 2.6594, cl 3.4389) | val recall@5 0.2257 * | 222s train


LightGCL + time attention · epoch 3:   0%|          | 0/1736 [00:00<?, ?it/s]

...

[LightGCL + time attention] epoch  9 | lr 3e-05 | loss 5.6578 (rec 2.3299, cl 2.6914) | val recall@5 0.2293 | 222s train


LightGCL + time attention · epoch 10:   0%|          | 0/1736 [00:00<?, ?it/s]

[LightGCL + time attention] epoch 10 | lr 9e-06 | loss 5.6513 (rec 2.3261, cl 2.6900) | val recall@5 0.2293 | 222s train
Đã lưu tham số epoch 7 vào data/runs/lightgcl_final_timeattn/model.pt


,lr,loss,loss_r,loss_s,recall@5,recall@20,ndcg@5,ndcg@20,map@5,map@20
epoch,,,,,,,,,,
1,"0,0001","10,7164","2,9036","7,5946","0,2185","0,5037","0,165","0,2544","0,1287","0,1613"
2,"0,0001","6,5253","2,6594","3,4389","0,2257","0,5164","0,1709","0,262","0,1337","0,1672"
3,"0,0001","5,7797","2,4438","2,7361","0,2273","0,5189","0,1721","0,2637","0,1348","0,1686"
4,"0,0001","5,7016","2,3749","2,7013","0,2281","0,5207","0,1729","0,2649","0,1355","0,1695"
5,"0,0001","5,6845","2,3553","2,6976","0,2289","0,5215","0,1734","0,2654","0,1358","0,1698"
6,"0,0001","5,6785","2,3469","2,6962","0,2291","0,522","0,1736","0,2657","0,136","0,17"
7,"0,0001","5,6757","2,3425","2,6955","0,2295","0,5227","0,1738","0,2659","0,1361","0,1701"
8,"0,0001","5,6741","2,3401","2,695","0,2292","0,5231","0,1736","0,2659","0,1359","0,1701"
9,0,"5,6578","2,3299","2,6914","0,2293","0,5227","0,1737","0,2659","0,136","0,1701"


,epoch tốt nhất,val recall@5,epoch đã train,giây train / epoch,GPU đỉnh (GiB)
LightGCL + time attention,7,"0,2295",10,"222,0218","6,5888"


## 3. Kết quả trên test

Cả ba model LightGCL chỉ được train trên booking trước 09/2014: hai model cũ (cạnh 0–1, cạnh theo số lần book) lấy từ checkpoint của LightGCL_Click.ipynb, LightGCL + time attention là model ở mục 2.

Khi chấm test, mọi booking 09–10/2014 được thêm vào graph (fold-in đến 10/2014): ma trận kề được chuẩn hoá lại (với time attention, thời điểm book gần nhất của mỗi cặp và softmax time attention cũng được tính lại trên graph mới), rồi embedding được truyền lại với tham số đã train, không train lại. 

Popular và History dùng dữ liệu tới 10/2014. User không có trong graph nhận danh sách Popular. Do đó với user mới, mọi phương pháp cho cùng một kết quả.

In [16]:
@torch.no_grad()
def fold_in(model, graph):
    """Graph-view embeddings of a trained model propagated over graph with the graph's own adjacency (adjacency(graph,
    model.edges)): the trained E_i0, the trained E_u0 for the users the model has seen and 0 for the others; nothing is
    trained."""
    adj_u, adj_i = (to_torch_sparse(a.tocoo(), DEVICE) for a in adjacency(graph, model.edges))
    E_u0 = torch.zeros(graph.n_users, model.state["E_u_0"].shape[1], device=DEVICE)
    rows = graph.user_index.get_indexer(model.user_ids)
    assert (rows >= 0).all(), "every user of the model must stay in the graph"
    E_u0[torch.as_tensor(rows, device=DEVICE)] = model.state["E_u_0"].to(DEVICE)
    E_u, E_i = [E_u0], [model.state["E_i_0"].to(DEVICE)]
    for _ in range(cfg.n_layers):  # the propagation of LightGCL.graph_layers
        E_u.append(torch.spmm(adj_u, E_i[-1]))
        E_i.append(torch.spmm(adj_i, E_u[-2]))
    return sum(E_u), sum(E_i)


def fold_in_scorer(model, graph):
    E_u, E_i = fold_in(model, graph)
    return with_fallback(
        lambda user_ids: E_u[torch.as_tensor(graph.user_index.get_indexer(user_ids), device=DEVICE)] @ E_i.T, graph)


# mmap: only the states used here are read from the checkpoints of LightGCL_Click.ipynb
checkpoints = {run: torch.load(PROJECT_ROOT / "data" / "runs" / run / "model.pt", map_location="cpu", mmap=True,
                               weights_only=False)
               for run in {run for run, _, _ in cfg.old_models.values()}}
MODELS = {label: SimpleNamespace(state=checkpoints[run]["states"][variant], user_ids=checkpoints[run]["user_ids"][variant],
                                 edges=edges, val=checkpoints[run]["results"][variant][f"val {cfg.main_metric}"])
          for label, (run, variant, edges) in cfg.old_models.items()}
MODELS["LightGCL + time attention"] = SimpleNamespace(state=saved["state"], user_ids=saved["user_ids"], edges="time",
                                                      val=saved["result"][f"val {cfg.main_metric}"])
del checkpoints

# Check: over the training graph, fold-in gives back the validation score each model had when trained.
display(fmt_table(pd.DataFrame({
    label: {f"val {cfg.main_metric} lúc train": m.val,
            "fold-in trên graph train": evaluate(fold_in_scorer(m, GRAPHS["train"]), val_split,
                                                 GRAPHS["train"].user_index)[cfg.main_metric]}
    for label, m in MODELS.items()
}).T))

scorers = {"Popular": popular(GRAPHS["train+val"]), "History": history(GRAPHS["train+val"]),
           **{label: fold_in_scorer(m, GRAPHS["train+val"]) for label, m in MODELS.items()}}
test_results = pd.concat({group: pd.DataFrame({name: evaluate(fn, test_split, users) for name, fn in scorers.items()}).T
                          for group, users in USER_GROUPS.items()}, names=["nhóm", "model"])
test_results.to_csv(run_dir / "test_results.csv")
for group in ("Tất cả user test", "User cũ (đã book trước 11/2014)", "User mới (chưa book trước 11/2014)"):
    print(f"{group}: {fmt_num(group_sizes[group])} user test")
    display(fmt_table(test_results.loc[group, REPORT]))


,val recall@5 lúc train,fold-in trên graph train
"LightGCL, cạnh 0–1","0,2102","0,2102"
"LightGCL, cạnh theo số lần book","0,216","0,216"
LightGCL + time attention,"0,2295","0,2295"


Tất cả user test: 222.688 user test


,recall@5,recall@20,ndcg@5,ndcg@20,map@5,map@20
model,,,,,,
Popular,"0,1459","0,3915","0,1019","0,1766","0,0778","0,103"
History,"0,199","0,4452","0,1467","0,2222","0,1155","0,1421"
"LightGCL, cạnh 0–1","0,1846","0,4466","0,1333","0,214","0,1034","0,1323"
"LightGCL, cạnh theo số lần book","0,1894","0,4511","0,1379","0,2185","0,1075","0,1364"
LightGCL + time attention,"0,2021","0,4769","0,1483","0,2329","0,1165","0,147"


User cũ (đã book trước 11/2014): 150.971 user test


,recall@5,recall@20,ndcg@5,ndcg@20,map@5,map@20
model,,,,,,
Popular,"0,1523","0,4047","0,1082","0,1857","0,0824","0,1087"
History,"0,2308","0,484","0,1742","0,2529","0,138","0,1663"
"LightGCL, cạnh 0–1","0,2095","0,486","0,1544","0,2408","0,1203","0,1518"
"LightGCL, cạnh theo số lần book","0,2166","0,4927","0,1612","0,2474","0,1263","0,1579"
LightGCL + time attention,"0,2353","0,5307","0,1765","0,2687","0,1395","0,1735"


User mới (chưa book trước 11/2014): 71.717 user test


,recall@5,recall@20,ndcg@5,ndcg@20,map@5,map@20
model,,,,,,
Popular,"0,1323","0,3636","0,0888","0,1576","0,068","0,0911"
History,"0,1323","0,3636","0,0888","0,1576","0,068","0,0911"
"LightGCL, cạnh 0–1","0,1323","0,3636","0,0888","0,1576","0,068","0,0911"
"LightGCL, cạnh theo số lần book","0,1323","0,3636","0,0888","0,1576","0,068","0,0911"
LightGCL + time attention,"0,1323","0,3636","0,0888","0,1576","0,068","0,0911"
